# MedCLIPSeg++ — six-stage pipeline walkthrough

This notebook demonstrates how an ultrasound case is routed through the six software stages. It is a software walkthrough, not a live clinical inference interface.

> **Execution status:** Stage 3 defaults to a synthetic MockModelAdapter. It returns a fixed ellipse and entropy pattern independent of the input image. The MedCLIPSegAdapter is not integrated with the current precomputed-result service; selecting it now raises an explicit error instead of silently using mock output. Stage 5 may also use demonstration metadata or generated embeddings if real gallery artifacts are unavailable.

Triage and retrieval outputs from the default run demonstrate orchestration only. They are not measurements on the input scan, model predictions, clinical evidence, or validation.

## What each stage demonstrates
| Stage | Demonstration | Limitation |
|---|---|---|
| 1. Inputs | Load image and request | Input validation |
| 2. Intent | Route request to segmentation/retrieval | Deterministic routing |
| 3. Segmentation | Run selected adapter | Default is synthetic mock |
| 4. Triage | Apply frozen policy to adapter output | Mock-derived decisions are illustrative |
| 5. Retrieval | Query PABP gallery | Verify manifest and embedding provenance |
| 6. Output | Render research review panel | Not a clinical report |

### Run
1. Set INPUT_FOLDER in Cell 2 to a folder with PNG or JPG images.
2. Keep MODEL_CHOICE set to Mock for the explicitly synthetic walkthrough.
3. Run cells in order. The notebook does not create synthetic scan files when the folder is empty.

For model, checkpoint, method, threshold, and experimental rationale, see Notebook 02 and the research documents linked there.


In [ ]:
# Cell 1: Environment Setup & 6-Stage Modular Imports
import sys
import os
import glob
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Configure Matplotlib backend for headless environments
import matplotlib
matplotlib.use('Agg')

# Resolve repository root
repo_root = Path.cwd().resolve()
if not (repo_root / "stages").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Import the 6 modular stages
from stages.stage1_inputs import load_clinical_inputs, ClinicalInputs
from stages.stage2_intent_router import parse_clinical_intent, ExecutionPlan
from stages.stage3_segmentation_backbone import MockModelAdapter, MedCLIPSegAdapter
from stages.stage4_selective_prediction import (
    evaluate_decision_policy, compute_all_uncertainty_metrics,
    FROZEN_TAU_STAR, FROZEN_TAU_NEG,
)
from stages.stage5_similarity_retrieval import (
    PABPRetrievalEngine, VisualEmbeddingExtractor,
    load_or_build_gallery_manifest,
)
from stages.stage6_system_output import compose_system_output, generate_triage_summary_report

print(f"[OK] Successfully imported all 6 modular stages from: {repo_root / 'stages'}")


## Configuration: input folder and execution mode

Choose an image folder and review the execution mode. The default Mock adapter produces a fixed synthetic ellipse and entropy map independent of the input image; it is useful only to demonstrate the software flow. The current MedCLIPSeg adapter path is not yet integrated with the live inference API. Stage 5 likewise needs verified real manifest and embedding artifacts before its results can be interpreted as image retrieval.



In [ ]:
# Cell 2: Pipeline Configuration
# Resolve candidate image folders (checks local project datasets or custom folder)
candidates = [
    repo_root.parent / "data/raw/BUSI/Test_Folder/img",
    repo_root / "data/raw/BUSI/Test_Folder/img",
    repo_root / "data/demo_scans",
]
found_folder = next((str(c) for c in candidates if c.exists()), str(candidates[0]))

# 1. Path to input folder containing ultrasound images (.png, .jpg)
INPUT_FOLDER = found_folder

# 2. Clinical text instruction (R)
CLINICAL_PROMPT = "A breast ultrasound scan showing a circumscribed lesion. Retrieve top 5 similar cases."

# 3. Mock is the only currently reliable notebook mode; MedCLIPSeg adapter integration is pending.
MODEL_CHOICE = "Mock"  # Synthetic demo only; does not run MedCLIPSeg inference.

# 4. Maximum number of scans to process in this run
MAX_SCANS_TO_PROCESS = 4

print(f"[CONFIG] Input Folder: {INPUT_FOLDER}")
print(f"[CONFIG] Clinical Prompt: {CLINICAL_PROMPT}")
print(f"[CONFIG] Model Adapter: {MODEL_CHOICE}")


---
## Initialize the demonstration pipeline

Builds an index from available gallery assets and initializes Stage 3. The output reports basic artifact availability; this is not proof that each retrieval artifact is scientifically valid.


In [ ]:
# Cell 3: Infrastructure Initialization
print("[INFO] Initializing Stage 3 Model Adapter...")
if MODEL_CHOICE == "MedCLIPSeg":
    raise NotImplementedError(
        "Live MedCLIPSeg inference is not integrated in this notebook. "
        "Use Mock only for the explicitly synthetic software walkthrough."
    )
else:
    model_adapter = MockModelAdapter()
    print("  -> DEMONSTRATION ONLY: using synthetic MockModelAdapter output.")

print("\n[INFO] Initializing Stage 5 Historical Gallery & PABP Index...")
gallery_records = load_or_build_gallery_manifest()
embedding_extractor = VisualEmbeddingExtractor()
gallery_embeddings = embedding_extractor.extract_or_load_gallery_embeddings(gallery_records)

pabp_engine = PABPRetrievalEngine(embedding_dim=512, nlist=16, nprobe=4, block_size=16)
pabp_engine.build_index(gallery_records, gallery_embeddings)

gallery_images_present = all(Path(r["image_path"]).is_file() for r in gallery_records)
embedding_model_available = getattr(embedding_extractor, "_extractor", None) is not None
print(f"  -> Indexed {len(gallery_records)} records into FAISS-IVF PABP engine.")
print(f"  -> Gallery image paths: {'present' if gallery_images_present else 'missing or synthetic'}")
print(f"  -> Embedding model wrapper: {'available; verify cache/model provenance' if embedding_model_available else 'synthetic fallback active'}")
print(f"  -> Gallery embeddings shape: {gallery_embeddings.shape}")
print("[OK] Software demonstration initialized. Stage 3 uses synthetic output; Stage 5 provenance is reported above.")


---
## Execute the six-stage software walkthrough

Each stage demonstrates software control flow only. With the default mock, mask and entropy are synthetic and independent of the scan. Applying the research thresholds to mock outputs does not reproduce their calibration study.

1. Stage 1: load and validate image X and request R.
2. Stage 2: parse R into an execution plan.
3. Stage 3: generate mask M and entropy U; default adapter is synthetic Mock.
4. Stage 4: apply the frozen policy; mock-derived decisions are illustrative only.
5. Stage 5: query available index; interpret only after verifying real assets.
6. Stage 6: render a research review visualization, not a clinical report.


In [ ]:
# Cell 4: Automated Execution Across All Input Images — 6-Stage Pipeline

# Find input image files
image_files = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.png")) + glob.glob(os.path.join(INPUT_FOLDER, "*.jpg")))

if not image_files:
    raise FileNotFoundError(
        f"No .png or .jpg images found in INPUT_FOLDER={INPUT_FOLDER!r}. "
        "Set INPUT_FOLDER in Cell 2 to a directory containing real input images. "
        "This pipeline notebook does not generate synthetic scan files."
    )

image_files = image_files[:MAX_SCANS_TO_PROCESS]
print(f"[INFO] Processing {len(image_files)} scans through the 6-stage pipeline...\n")

batch_telemetry = []

for idx, img_path in enumerate(image_files):
    case_id = Path(img_path).name
    print(f"------------------------------------------------------------------------")
    print(f"Processing Case [{idx+1}/{len(image_files)}]: {case_id}")
    
    # --- STAGE 1: Load & Validate Inputs (X, R) ---
    clinical_inputs = load_clinical_inputs(img_path, CLINICAL_PROMPT)
    print(f"  -> STAGE 1 [INPUTS]: Image loaded ({clinical_inputs.original_size}), Prompt: '{clinical_inputs.prompt[:50]}...'")
    
    # --- STAGE 2: Intent Router -> Execution Plan (s, r, K) ---
    plan = parse_clinical_intent(clinical_inputs.prompt)
    print(f"  -> STAGE 2 [INTENT ROUTER]: s={plan.s}, r={plan.r}, K={plan.k}, Mode='{plan.mode}'")
    
    # --- STAGE 3: MedCLIPSeg Segmentation -> M, U ---
    binary_mask, entropy_map = None, None
    if plan.s == 1:
        raw_img = Image.open(img_path).convert("RGB")
        binary_mask, entropy_map = model_adapter.predict_with_uncertainty(raw_img, plan.target_concept)
        fg_px = int(np.sum(binary_mask > 0))
        print(f"  -> STAGE 3 [MOCK DEMO]: Mask M generated ({fg_px} fg pixels), Uncertainty U computed.")
    else:
        raw_img = Image.open(img_path).convert("RGB")
        print(f"  -> STAGE 3 [MOCK DEMO]: Skipped (s=0, retrieve-only mode).")
    
    # --- STAGE 4: Selective Prediction (Triage) -> Gate g ---
    decision = None
    if plan.s == 1 and binary_mask is not None:
        decision = evaluate_decision_policy(binary_mask, entropy_map)
        print(f"  -> STAGE 4 [TRIAGE]: Status={decision.status}, g={decision.g}, "
              f"U_fg={decision.u_fg:.5f}, Margin={decision.margin:+.5f}")
    else:
        # Retrieve-only: no triage needed, gate is open by default
        print(f"  -> STAGE 4 [TRIAGE]: Skipped (retrieve-only mode, g=1 by default).")
    
    # --- STAGE 5: Similarity Retrieval (gated by g AND r) ---
    retrieved_neighbors = []
    pabp_metrics = None
    g = decision.g if decision else 1  # Default g=1 for retrieve-only
    if g == 1 and plan.r == 1:
        query_vec = embedding_extractor.extract_image_embedding(raw_img)
        retrieved_neighbors, pabp_metrics = pabp_engine.search_top_k(query_vec, k=plan.k)
        print(f"  -> STAGE 5 [RETRIEVAL]: Found {len(retrieved_neighbors)} similar cases. "
              f"Blocks pruned: {pabp_metrics.blocks_pruned_pabp} | Distances: {pabp_metrics.vector_distances_computed}")
    elif plan.r == 0:
        print(f"  -> STAGE 5 [RETRIEVAL]: Skipped (r=0, segment-only mode).")
    else:
        print(f"  -> STAGE 5 [RETRIEVAL]: Blocked (g=0, flagged for review). Diverted to review queue.")

    # --- STAGE 6: System Output ---
    fig = compose_system_output(
        s=plan.s,
        r=plan.r,
        raw_image=raw_img,
        binary_mask=binary_mask,
        entropy_map=entropy_map,
        decision_outcome=decision,
        retrieved_neighbors=retrieved_neighbors,
    )
    
    try:
        from IPython.display import display
        display(fig)
    except Exception:
        pass
    plt.close(fig)

    # Record telemetry
    batch_telemetry.append({
        "image_id": case_id,
        "s": plan.s,
        "r": plan.r,
        "g": g,
        "status": decision.status if decision else "RETRIEVE_ONLY",
        "u_fg": decision.u_fg if decision else float("nan"),
        "margin": decision.margin if decision else float("nan"),
        "fg_pixels": decision.fg_pixels if decision else 0,
        "k_retrieved": len(retrieved_neighbors),
    })

print(f"------------------------------------------------------------------------")
print(f"[OK] Batch processing complete for {len(image_files)} cases.")


---
## Stage 6: demonstration summary
Shows the run-level triage and retrieval fields. With the default mock mode, all decisions are synthetic examples, not evaluated predictions.


In [ ]:
# Cell 5: Demonstration-only summary based on synthetic mock outputs
df_report = generate_triage_summary_report(batch_telemetry)

print("="*90)
print(" MEDCLIPSEG++ 6-STAGE SOFTWARE WALKTHROUGH SUMMARY")
print("="*90)
print(df_report.to_string(index=False))
print("="*90)
